In [1]:
import glob
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings('ignore')

# 1. Instant Competition Root Resolution (Zero disk traversal overhead)
POSSIBLE_COMP_DIRS = [
    '/kaggle/input/competitions/rsna-knee-abnormality-detection',
    '/kaggle/input/rsna-knee-abnormality-detection'
]
COMP_DIR = None
for p in POSSIBLE_COMP_DIRS:
    if os.path.exists(os.path.join(p, 'train.csv')) and os.path.exists(os.path.join(p, 'train_series.csv')):
        COMP_DIR = p
        break

assert COMP_DIR is not None, 'Competition directory not found in /kaggle/input.'
print(f'Mounted COMP_DIR: {COMP_DIR}')

df_train = pd.read_csv(os.path.join(COMP_DIR, 'train.csv'))
df_series = pd.read_csv(os.path.join(COMP_DIR, 'train_series.csv'))

# 2. Strict prioritization for v4_blend LLM labels
POSSIBLE_LLM_FOLDERS = [
    '/kaggle/input/datasets/stevenleehans/rsna-knee-llm-report-labels',
    '/kaggle/input/rsna-knee-llm-report-labels'
]
llm_folder = None
for fld in POSSIBLE_LLM_FOLDERS:
    if os.path.exists(fld):
        llm_folder = fld
        break

assert llm_folder is not None, 'Failed to locate LLM labels directory.'
available_csvs = [f for f in os.listdir(llm_folder) if f.endswith('.csv')]
print(f'Available CSVs in LLM dataset: {available_csvs}')

preferred_order = [
    'llm_labels_v4_blend.csv',
    'llm_labels_v4.csv',
    'llm_labels_v3.csv',
    'llm_labels_v2.csv',
]
LLM_PATH = None

for fname in preferred_order:
    target_file = os.path.join(llm_folder, fname)
    if os.path.exists(target_file):
        LLM_PATH = target_file
        break

if LLM_PATH is None and len(available_csvs) > 0:
    LLM_PATH = os.path.join(llm_folder, available_csvs[0])

assert LLM_PATH is not None, 'Failed to locate any valid LLM CSV.'
print(f'Selected LLM source: {LLM_PATH}')

df_llm = pd.read_csv(LLM_PATH)

# 3. Explicitly lock the 12 medical abnormality targets
TARGET_COLS = [
    'ACL',
    'MCL',
    'Medial Meniscus',
    'Lateral Meniscus',
    'Medial OA',
    'Lateral OA',
    'PF OA',
    'Effusion',
    'Synovitis',
    "Baker's",
    'Contusion',
    'Fracture',
]

# Ensure targets exist in LLM file
for col in TARGET_COLS:
    assert (
        col in df_llm.columns
    ), f"Expected target '{col}' missing from {os.path.basename(LLM_PATH)}"

# Merge weak LLM labels into df_train, dropping non-label metadata like 'Report'
llm_subset = df_llm[['StudyInstanceUID'] + TARGET_COLS].copy()
df_train = df_train.merge(
    llm_subset, on='StudyInstanceUID', how='left', suffixes=('', '_llm')
)

print(f'Total Studies: {len(df_train)}')
print(f'Verified Target Columns ({len(TARGET_COLS)}): {TARGET_COLS}')

Mounted COMP_DIR: /kaggle/input/competitions/rsna-knee-abnormality-detection
Available CSVs in LLM dataset: ['llm_labels_v2.csv', 'llm_labels_full.csv', 'llm_labels_v4_blend.csv']
Selected LLM source: /kaggle/input/datasets/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv
Total Studies: 4407
Verified Target Columns (12): ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']


In [2]:
# 1. Consolidate official competition labels with weak LLM labels
is_gold = df_train[TARGET_COLS].notnull().all(axis=1)
df_train['is_gold'] = is_gold

for col in TARGET_COLS:
    # Fill missing official labels with the blended LLM weak labels
    llm_col = f"{col}_llm"
    if llm_col in df_train.columns:
        df_train[col] = df_train[col].combine_first(df_train[llm_col])
    
    # Cast targets to float32 for BCEWithLogitsLoss
    df_train[col] = df_train[col].astype(np.float32)

gold_count = int(is_gold.sum())
weak_count = len(df_train) - gold_count

print(f"Gold Ground-Truth Studies : {gold_count}")
print(f"Weak LLM-Labeled Studies  : {weak_count}")
assert df_train[TARGET_COLS].isnull().sum().sum() == 0, "Null values remain in consolidated targets."

# 2. Balanced 5-Fold Split (Distribute gold studies evenly across folds)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
df_train['fold'] = -1

# Split gold studies
gold_idx = df_train[df_train['is_gold']].index.to_numpy()
for fold, (_, val_i) in enumerate(kf.split(gold_idx)):
    df_train.loc[gold_idx[val_i], 'fold'] = fold

# Split weak studies
weak_idx = df_train[~df_train['is_gold']].index.to_numpy()
for fold, (_, val_i) in enumerate(kf.split(weak_idx)):
    df_train.loc[weak_idx[val_i], 'fold'] = fold

print("\nFold Distribution (Total / Gold):")
for f in range(5):
    f_total = (df_train['fold'] == f).sum()
    f_gold = ((df_train['fold'] == f) & (df_train['is_gold'])).sum()
    print(f"  Fold {f}: {f_total} total studies | {f_gold} gold studies")

Gold Ground-Truth Studies : 58
Weak LLM-Labeled Studies  : 4349

Fold Distribution (Total / Gold):
  Fold 0: 882 total studies | 12 gold studies
  Fold 1: 882 total studies | 12 gold studies
  Fold 2: 882 total studies | 12 gold studies
  Fold 3: 881 total studies | 11 gold studies
  Fold 4: 880 total studies | 11 gold studies


In [3]:
def find_cache_dir(dataset_hint):
    """Recursively searches /kaggle/input for the folder containing .pt files."""
    for root, _, files in os.walk('/kaggle/input'):
        if dataset_hint.lower() in root.lower():
            pt_files = [f for f in files if f.endswith('.pt')]
            if len(pt_files) > 0:
                return root, len(pt_files)
    return None, 0

SAG_DIR, sag_count = find_cache_dir('sagittal')
COR_DIR, cor_count = find_cache_dir('coronal')

print("=" * 65)
print(f"Sagittal Cache Path : {SAG_DIR}")
print(f"Sagittal .pt Count  : {sag_count}")
print("-" * 65)
print(f"Coronal Cache Path  : {COR_DIR}")
print(f"Coronal .pt Count   : {cor_count}")
print("=" * 65)

assert SAG_DIR is not None and sag_count > 0, "Sagittal cache directory not found or empty."
assert COR_DIR is not None and cor_count > 0, "Coronal cache directory not found or empty."

Sagittal Cache Path : /kaggle/input/datasets/pupoolapoju/rsna-2026-25d-sagittal-cache-256/rsna_25d_cache
Sagittal .pt Count  : 4407
-----------------------------------------------------------------
Coronal Cache Path  : /kaggle/input/datasets/pupoolapoju/rsna-coronal-adjacent-cache-256
Coronal .pt Count   : 4407


In [4]:
import torchvision.transforms.functional as TF
import random

class DualPlaneDataset(Dataset):
    def __init__(self, df, sag_dir, cor_dir, target_cols, is_train=True):
        self.df = df.reset_index(drop=True)
        self.sag_dir = sag_dir
        self.cor_dir = cor_dir
        self.target_cols = target_cols
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def _apply_augmentations(self, sag, cor):
        # Affine: slight translation and scale (NO HORIZONTAL FLIP)
        if random.random() > 0.5:
            angle = random.uniform(-10, 10)
            translate = (random.uniform(-0.05, 0.05) * 256, random.uniform(-0.05, 0.05) * 256)
            scale = random.uniform(0.95, 1.05)
            
            sag = TF.affine(sag, angle=angle, translate=list(translate), scale=scale, shear=0)
            cor = TF.affine(cor, angle=angle, translate=list(translate), scale=scale, shear=0)
            
        return sag, cor

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_id = str(row['StudyInstanceUID'])
        
        # 1. Load Sagittal
        sag_path = os.path.join(self.sag_dir, f"{study_id}.pt")
        if os.path.exists(sag_path):
            data = torch.load(sag_path, map_location='cpu')
            sag_tensor = data['image'] if isinstance(data, dict) and 'image' in data else data
        else:
            sag_tensor = torch.zeros((5, 256, 256), dtype=torch.float32)

        # 2. Load Coronal
        cor_path = os.path.join(self.cor_dir, f"{study_id}.pt")
        if os.path.exists(cor_path):
            data = torch.load(cor_path, map_location='cpu')
            cor_tensor = data['image'] if isinstance(data, dict) and 'image' in data else data
        else:
            cor_tensor = torch.zeros((5, 256, 256), dtype=torch.float32)

        # 3. Augmentations
        if self.is_train:
            sag_tensor, cor_tensor = self._apply_augmentations(sag_tensor, cor_tensor)

        # 4. Targets
        labels = torch.tensor(row[self.target_cols].values.astype(np.float32), dtype=torch.float32)

        return {
            'sag': sag_tensor,
            'cor': cor_tensor,
            'targets': labels,
            'study_id': study_id
        }

In [5]:
import torchvision.models as models

def get_adapted_backbone(weights_path=None):
    model = models.efficientnet_b0(weights=None)
    
    # Load offline pretrained weights if available
    if weights_path and os.path.exists(weights_path):
        state_dict = torch.load(weights_path, map_location='cpu')
        model.load_state_dict(state_dict)
        print(f"Loaded weights from {weights_path}")
        
    # Adapt first conv layer from 3 channels to 5 channels
    orig_conv = model.features[0][0]
    new_conv = nn.Conv2d(
        in_channels=5,
        out_channels=orig_conv.out_channels,
        kernel_size=orig_conv.kernel_size,
        stride=orig_conv.stride,
        padding=orig_conv.padding,
        bias=False
    )
    
    with torch.no_grad():
        # Copy 3 channels and duplicate/average for channels 4 and 5
        new_conv.weight[:, :3, :, :] = orig_conv.weight
        new_conv.weight[:, 3:, :, :] = orig_conv.weight[:, :2, :, :]
        
    model.features[0][0] = new_conv
    
    # Strip classifier head; output pooled 1280-d features
    features_extractor = nn.Sequential(
        model.features,
        model.avgpool,
        nn.Flatten()
    )
    return features_extractor

class DualBranchKneeNet(nn.Module):
    def __init__(self, num_classes=12, weights_path=None):
        super().__init__()
        self.sag_branch = get_adapted_backbone(weights_path)
        self.cor_branch = get_adapted_backbone(weights_path)
        
        # Late fusion: 1280 (Sagittal) + 1280 (Coronal) = 2560
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(1280 * 2, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(p=0.2),
            nn.Linear(512, num_classes)
        )

    def forward(self, sag, cor):
        feat_sag = self.sag_branch(sag)
        feat_cor = self.cor_branch(cor)
        
        fused = torch.cat([feat_sag, feat_cor], dim=1)
        logits = self.classifier(fused)
        return logits

# Locate offline weights file
WEIGHTS_PATH = None
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        if 'efficientnet-b0' in f.lower() or 'efficientnet_b0' in f.lower():
            if f.endswith('.pth') or f.endswith('.pt'):
                WEIGHTS_PATH = os.path.join(root, f)
                break
    if WEIGHTS_PATH:
        break

print(f"Detected EfficientNet-B0 weights path: {WEIGHTS_PATH}")
test_model = DualBranchKneeNet(num_classes=12, weights_path=WEIGHTS_PATH)
dummy_in = torch.randn(2, 5, 256, 256)
out = test_model(dummy_in, dummy_in)
print(f"Forward pass output shape: {out.shape}")
assert out.shape == (2, 12), f"Expected shape (2, 12), got {out.shape}"

Detected EfficientNet-B0 weights path: /kaggle/input/datasets/pupoolapoju/rsna-efficientnet-b0-weights/efficientnet_b0_imagenet.pth
Loaded weights from /kaggle/input/datasets/pupoolapoju/rsna-efficientnet-b0-weights/efficientnet_b0_imagenet.pth
Loaded weights from /kaggle/input/datasets/pupoolapoju/rsna-efficientnet-b0-weights/efficientnet_b0_imagenet.pth
Forward pass output shape: torch.Size([2, 12])


In [6]:
from sklearn.metrics import roc_auc_score
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert torch.cuda.is_available(), "GPU T4 accelerator not active in Session Options."

# 1. Hyperparameters
FOLD = 0
EPOCHS = 5
BATCH_SIZE = 16
LR = 3e-4
WEIGHT_DECAY = 1e-2
NUM_CLASSES = len(TARGET_COLS)

# 2. Split DataFrames
train_df = df_train[df_train['fold'] != FOLD].reset_index(drop=True)
val_df = df_train[df_train['fold'] == FOLD].reset_index(drop=True)

train_loader = DataLoader(
    DualPlaneDataset(train_df, SAG_DIR, COR_DIR, TARGET_COLS, is_train=True),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    DualPlaneDataset(val_df, SAG_DIR, COR_DIR, TARGET_COLS, is_train=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# 3. Model, Criterion, Optimizer, Scheduler
model = DualBranchKneeNet(num_classes=NUM_CLASSES, weights_path=WEIGHTS_PATH).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = GradScaler()

def evaluate_predictions(y_true, y_prob, target_cols, threshold=0.5):
    """
    Binarizes continuous pseudo-targets at threshold (0.5) for ROC-AUC evaluation.
    Safely ignores single-class targets.
    """
    y_true_bin = np.where(y_true >= threshold, 1, 0)
    class_aucs = []
    
    for i in range(len(target_cols)):
        y_t = y_true_bin[:, i]
        y_p = y_prob[:, i]
        
        # ROC-AUC requires at least one positive and one negative sample
        if len(np.unique(y_t)) > 1:
            try:
                auc = roc_auc_score(y_t, y_p)
                class_aucs.append(auc)
            except ValueError:
                class_aucs.append(np.nan)
        else:
            class_aucs.append(np.nan)
            
    # Calculate macro average across classes that had valid splits
    valid_aucs = [a for a in class_aucs if not np.isnan(a)]
    macro_auc = np.mean(valid_aucs) if len(valid_aucs) > 0 else 0.5
    return macro_auc, class_aucs

# 4. Training Loop
best_macro_auc = 0.0
val_is_gold = val_df['is_gold'].to_numpy()

print(f"Starting Training: Fold {FOLD} | Train Studies: {len(train_df)} | Val Studies: {len(val_df)} | Device: {DEVICE}")
print("-" * 80)

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    
    # --- Train Phase ---
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        sag = batch['sag'].to(DEVICE, non_blocking=True)
        cor = batch['cor'].to(DEVICE, non_blocking=True)
        targets = batch['targets'].to(DEVICE, non_blocking=True)
        
        optimizer.zero_grad()
        with autocast():
            logits = model(sag, cor)
            loss = criterion(logits, targets)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * sag.size(0)
        
    scheduler.step()
    train_loss = running_loss / len(train_df)
    
    # --- Validation Phase ---
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in val_loader:
            sag = batch['sag'].to(DEVICE, non_blocking=True)
            cor = batch['cor'].to(DEVICE, non_blocking=True)
            targets = batch['targets'].to(DEVICE, non_blocking=True)
            
            with autocast():
                logits = model(sag, cor)
                loss = criterion(logits, targets)
                
            val_loss += loss.item() * sag.size(0)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(probs)
            all_targets.append(targets.cpu().numpy())
            
    val_loss = val_loss / len(val_df)
    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)
    
    # Compute Metrics
    overall_auc, _ = evaluate_predictions(y_true, y_pred, TARGET_COLS)
    gold_auc, _ = evaluate_predictions(y_true[val_is_gold], y_pred[val_is_gold], TARGET_COLS)
    
    elapsed = time.time() - start_time
    is_best = overall_auc > best_macro_auc
    if is_best:
        best_macro_auc = overall_auc
        torch.save(model.state_dict(), f"rsna_v8_dual_branch_fold{FOLD}_best.pth")
        save_mark = "[BEST SAVED]"
    else:
        save_mark = ""
        
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} [{elapsed:.0f}s] | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Overall Val AUC: {overall_auc:.4f} | "
          f"Gold Val AUC: {gold_auc:.4f} {save_mark}")

print("-" * 80)
print(f"Training Complete. Peak Overall Val Macro-AUC: {best_macro_auc:.4f}")

Loaded weights from /kaggle/input/datasets/pupoolapoju/rsna-efficientnet-b0-weights/efficientnet_b0_imagenet.pth
Loaded weights from /kaggle/input/datasets/pupoolapoju/rsna-efficientnet-b0-weights/efficientnet_b0_imagenet.pth
Starting Training: Fold 0 | Train Studies: 3525 | Val Studies: 882 | Device: cuda
--------------------------------------------------------------------------------
Epoch 01/05 [131s] | Train Loss: 0.5966 | Val Loss: 0.5605 | Overall Val AUC: 0.7222 | Gold Val AUC: 0.6571 [BEST SAVED]
Epoch 02/05 [36s] | Train Loss: 0.5434 | Val Loss: 0.5526 | Overall Val AUC: 0.7471 | Gold Val AUC: 0.7066 [BEST SAVED]
Epoch 03/05 [35s] | Train Loss: 0.5113 | Val Loss: 0.5422 | Overall Val AUC: 0.7639 | Gold Val AUC: 0.7625 [BEST SAVED]
Epoch 04/05 [35s] | Train Loss: 0.4837 | Val Loss: 0.5416 | Overall Val AUC: 0.7747 | Gold Val AUC: 0.7200 [BEST SAVED]
Epoch 05/05 [35s] | Train Loss: 0.4624 | Val Loss: 0.5392 | Overall Val AUC: 0.7761 | Gold Val AUC: 0.7425 [BEST SAVED]
----------

In [7]:
# Load the best saved Fold 0 checkpoint
best_model = DualBranchKneeNet(num_classes=NUM_CLASSES, weights_path=None).to(DEVICE)
best_model.load_state_dict(torch.load(f"rsna_v8_dual_branch_fold{FOLD}_best.pth"))
best_model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for batch in val_loader:
        sag = batch['sag'].to(DEVICE, non_blocking=True)
        cor = batch['cor'].to(DEVICE, non_blocking=True)
        with autocast():
            logits = best_model(sag, cor)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_preds.append(probs)
        all_targets.append(batch['targets'].numpy())

y_pred = np.vstack(all_preds)
y_true = np.vstack(all_targets)
y_true_bin = np.where(y_true >= 0.5, 1, 0)

print(f"{'Class Name':<20} | {'Positives':<10} | {'ROC-AUC':<10}")
print("-" * 46)

class_scores = []
for i, col in enumerate(TARGET_COLS):
    yt = y_true_bin[:, i]
    yp = y_pred[:, i]
    pos_count = int(yt.sum())
    
    if len(np.unique(yt)) > 1:
        auc = roc_auc_score(yt, yp)
        class_scores.append(auc)
        print(f"{col:<20} | {pos_count:<10} | {auc:.4f}")
    else:
        print(f"{col:<20} | {pos_count:<10} | N/A (No positives)")

print("-" * 46)
print(f"{'Macro-Average':<20} | {'--':<10} | {np.mean(class_scores):.4f}")

Class Name           | Positives  | ROC-AUC   
----------------------------------------------
ACL                  | 182        | 0.7955
MCL                  | 139        | 0.7235
Medial Meniscus      | 357        | 0.7703
Lateral Meniscus     | 132        | 0.6785
Medial OA            | 337        | 0.8333
Lateral OA           | 232        | 0.7689
PF OA                | 419        | 0.7827
Effusion             | 512        | 0.8200
Synovitis            | 105        | 0.8619
Baker's              | 225        | 0.7067
Contusion            | 156        | 0.7753
Fracture             | 70         | 0.7968
----------------------------------------------
Macro-Average        | --         | 0.7761


In [8]:
import os
import pydicom
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
from torch.cuda.amp import autocast

# 1. Path Verification
TEST_SERIES_PATH = os.path.join(COMP_DIR, 'test_series.csv')
TEST_DICOM_DIR = os.path.join(COMP_DIR, 'test_series')
SAMPLE_SUB_PATH = os.path.join(COMP_DIR, 'sample_submission.csv')
CHECKPOINT_PATH = f"rsna_v8_dual_branch_fold{FOLD}_best.pth"

assert os.path.exists(CHECKPOINT_PATH), f"Fatal: Checkpoint {CHECKPOINT_PATH} not found on disk."
assert os.path.exists(TEST_SERIES_PATH), f"Fatal: test_series.csv not found at {TEST_SERIES_PATH}."
assert os.path.exists(TEST_DICOM_DIR), f"Fatal: DICOM directory not found at {TEST_DICOM_DIR}."

df_test_series = pd.read_csv(TEST_SERIES_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
unique_test_studies = df_test_series['StudyInstanceUID'].unique()

print(f"Loaded test metadata. Total test studies to process: {len(unique_test_studies)}")

# 2. Self-Contained Model Initialization
best_model = DualBranchKneeNet(num_classes=len(TARGET_COLS), weights_path=None).to(DEVICE)
best_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
best_model.eval()
print(f"Successfully loaded model weights from {CHECKPOINT_PATH}")

# 3. Dynamic Slice Extraction
resize_256 = T.Resize((256, 256), antialias=True)

def process_raw_series(series_folder):
    if not os.path.exists(series_folder):
        return torch.zeros((5, 256, 256), dtype=torch.float32)
        
    dcm_files = [os.path.join(series_folder, f) for f in os.listdir(series_folder) if f.endswith('.dcm')]
    if len(dcm_files) == 0:
        return torch.zeros((5, 256, 256), dtype=torch.float32)
        
    slices = []
    for f in dcm_files:
        try:
            dcm = pydicom.dcmread(f, stop_before_pixels=True)
            pos = float(dcm.ImagePositionPatient[2]) if hasattr(dcm, 'ImagePositionPatient') else float(dcm.InstanceNumber)
            slices.append((pos, f))
        except:
            continue
            
    if len(slices) == 0:
        return torch.zeros((5, 256, 256), dtype=torch.float32)
        
    slices.sort(key=lambda x: x[0])
    num_slices = len(slices)
    
    # Extract 5 contiguous adjacent central slices
    center_idx = num_slices // 2
    indices = [max(0, min(num_slices - 1, center_idx + offset)) for offset in [-2, -1, 0, 1, 2]]
    
    arrays = []
    for idx in indices:
        try:
            dcm = pydicom.dcmread(slices[idx][1])
            arr = dcm.pixel_array.astype(np.float32)
            arrays.append(arr)
        except:
            arrays.append(np.zeros((256, 256), dtype=np.float32))
            
    vol = np.stack(arrays)
    lo, hi = vol.min(), vol.max()
    vol = (vol - lo) / (hi - lo) if hi > lo else np.zeros_like(vol)
    
    return resize_256(torch.tensor(vol, dtype=torch.float32))

# 4. Inference Execution Loop
test_records = []
non_zero_sags = 0
non_zero_cors = 0

with torch.no_grad():
    for study_id in unique_test_studies:
        study_sub = df_test_series[df_test_series['StudyInstanceUID'] == study_id]
        
        sag_rows = study_sub[study_sub['Anatomical_Plane'] == 'Sagittal']
        cor_rows = study_sub[study_sub['Anatomical_Plane'] == 'Coronal']
        
        # Load Sagittal Branch
        if len(sag_rows) > 0:
            sag_dir = os.path.join(TEST_DICOM_DIR, str(study_id), str(sag_rows.iloc[0]['SeriesInstanceUID']))
            sag_t = process_raw_series(sag_dir)
            if sag_t.abs().sum() > 0: non_zero_sags += 1
        else:
            sag_t = torch.zeros((5, 256, 256), dtype=torch.float32)
            
        # Load Coronal Branch
        if len(cor_rows) > 0:
            cor_dir = os.path.join(TEST_DICOM_DIR, str(study_id), str(cor_rows.iloc[0]['SeriesInstanceUID']))
            cor_t = process_raw_series(cor_dir)
            if cor_t.abs().sum() > 0: non_zero_cors += 1
        else:
            cor_t = torch.zeros((5, 256, 256), dtype=torch.float32)
            
        sag_t = sag_t.unsqueeze(0).to(DEVICE)
        cor_t = cor_t.unsqueeze(0).to(DEVICE)
        
        with autocast():
            logits = best_model(sag_t, cor_t)
            probs = torch.sigmoid(logits).cpu().numpy()[0]
            
        rec = {'StudyInstanceUID': study_id}
        for i, col in enumerate(TARGET_COLS):
            rec[col] = float(probs[i])
        test_records.append(rec)

print(f"\nExtraction Check:")
print(f"  Valid non-zero Sagittal series: {non_zero_sags}/{len(unique_test_studies)}")
print(f"  Valid non-zero Coronal series : {non_zero_cors}/{len(unique_test_studies)}")
assert non_zero_sags > 0 or non_zero_cors > 0, "Fatal: Pipeline extracted only zero tensors across all test inputs."

# 5. Build and Export submission.csv
sub_df = pd.DataFrame(test_records)

# Enforce exact column order from sample_submission.csv
sub_df = sub_df[sample_sub.columns]
sub_df.to_csv("submission.csv", index=False)

print("\n--- Verified Final Submission ---")
print(f"Output Shape: {sub_df.shape}")
print(sub_df.head())

assert sub_df.isnull().sum().sum() == 0, "Submission contains NaN/null values."
assert len(sub_df) == len(unique_test_studies), f"Expected {len(unique_test_studies)} rows, found {len(sub_df)}."
assert sub_df.shape[1] == 13, f"Expected 13 columns, found {sub_df.shape[1]}."
assert os.path.exists("submission.csv"), "submission.csv was not written to /kaggle/working."
print("\nValidation passed. Ready for Save & Run All commit.")

Loaded test metadata. Total test studies to process: 3
Successfully loaded model weights from rsna_v8_dual_branch_fold0_best.pth

Extraction Check:
  Valid non-zero Sagittal series: 3/3
  Valid non-zero Coronal series : 3/3

--- Verified Final Submission ---
Output Shape: (3, 13)
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.089111  0.151367   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.247070  0.160522   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.065002  0.064270   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.483887          0.179932   0.541504    0.327637  0.641113   
1         0.372314          0.390381   0.608398    0.567383  0.651855   
2         0.385498          0.316406   0.607422    0.546875  0.864746   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  0.619629   0.382812  0.409424   0.214478  0.191650  
1  0.4118